<a href="https://colab.research.google.com/github/vkshadoww/114-2-Programing-Language/blob/main/Copy_of_HW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_Part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

安裝必要的套件

In [126]:
!pip install -q google-generativeai

In [127]:
import gspread # Added for self-containment
from google.colab import auth # Added for self-containment
from google.auth import default # Added for self-containment
from datetime import datetime # Added for self-containment

In [128]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

from google.colab import userdata
from google import genai

# Global variables for Google Sheet connection
SHEET_URL = "https://docs.google.com/spreadsheets/d/1pA05rdBicdtbP1LZQIQrls8HxSqm74JgKmyFKAqLvzw/edit?usp=drivesdk"
WORKSHEET_NAME = "工作表2"
REQUIRED_COLUMNS = ["日期", "科目", "分數", "是否已訂正", "AI摘要"]

# Global headers and column index for checkbox validation
new_headers = ["日期", "科目", "分數", "是否已訂正", "AI摘要"]
corrected_col_index = new_headers.index('是否已訂正') + 1 # +1 because Sheets API column index starts from 1, Python list index from 0

### 步驟 2: 導入函式庫與設定 API 金鑰

設定 Google Sheet 連線

In [129]:
_gc = None
_ws = None

def setup_gspread(sheet_url, worksheet_name):
    global _gc, _ws
    if _gc is None or _ws is None:
        print("--- 正在進行 Google Sheet 身份驗證和連線... ---")
        try:
            auth.authenticate_user()
            creds, _ = default()
            _gc = gspread.authorize(creds)
            sh = _gc.open_by_url(sheet_url)
            _ws = sh.worksheet(worksheet_name)
            print("--- Google Sheet 連線成功。---")
        except Exception as e:
            print(f"Google Sheet 連線失敗：{e}")
            _gc = None
            _ws = None


In [130]:
# 確保 Google Sheet 連線已經建立
if _ws is None:
    setup_gspread(SHEET_URL, WORKSHEET_NAME)

if _ws is not None:
    print(f"--- 正在清除工作表 '{WORKSHEET_NAME}' 的內容並設定新標題... ---")
    _ws.clear() # 清除所有內容

    # 設定新的欄位名稱 (new_headers is now global)
    _ws.update(range_name='A1', values=[new_headers]) # 將新標題寫入第一行，使用命名引數修正 DeprecationWarning
    print("--- Google Sheet 內容已清除，新標題已設定。---")
else:
    print("--- Google Sheet 連線失敗，無法執行清除和設定標題操作。---")

--- 正在進行 Google Sheet 身份驗證和連線... ---
--- Google Sheet 連線成功。---
--- 正在清除工作表 '工作表2' 的內容並設定新標題... ---
--- Google Sheet 內容已清除，新標題已設定。---


In [131]:
from googleapiclient.discovery import build

def get_sheet_id_from_url(sheet_url):
    """
    從 Google Sheet URL 中提取工作表 ID。
    """
    import re
    match = re.search(r'/spreadsheets/d/([a-zA-Z0-9-_]+)', sheet_url)
    if match:
        return match.group(1)
    return None

def set_checkbox_data_validation(sheet_url, worksheet_name, column_index, header_row=1):
    """
    設定 Google Sheet 特定欄位的資料驗證，將 'TRUE'/'FALSE' 轉換為核取方塊。
    Args:
        sheet_url (str): Google Sheet 的 URL。
        worksheet_name (str): 工作表的名稱。
        column_index (int): 要設定核取方塊的欄位索引（例如，A 欄是 1，B 欄是 2）。
        header_row (int): 標題所在的列數，資料驗證會從標題下一列開始應用。
    """
    print(f"--- 正在嘗試為 '{worksheet_name}' 工作表的第 {column_index} 欄設定核取方塊資料驗證... ---")
    try:
        auth.authenticate_user()
        creds, _ = default(['https://www.googleapis.com/auth/spreadsheets'])
        service = build('sheets', 'v4', credentials=creds)

        sheet_id = get_sheet_id_from_url(sheet_url)
        if not sheet_id:
            print("錯誤：無法從提供的 URL 提取工作表 ID。")
            return

        # 獲取工作表的詳細資訊以找到 worksheet_id
        spreadsheet_metadata = service.spreadsheets().get(spreadsheetId=sheet_id).execute()
        worksheet_id = None
        for sheet_properties in spreadsheet_metadata.get('sheets', []):
            if sheet_properties['properties']['title'] == worksheet_name:
                worksheet_id = sheet_properties['properties']['sheetId']
                break

        if worksheet_id is None:
            print(f"錯誤：在試算表中找不到名稱為 '{worksheet_name}' 的工作表。")
            return

        # 建立設定資料驗證的請求
        requests = [
            {
                'setDataValidation': {
                    'range': {
                        'sheetId': worksheet_id,
                        'startRowIndex': header_row, # 從標題下一列開始
                        'endRowIndex': 10000, # 假設最大行數，您可以根據需要調整
                        'startColumnIndex': column_index - 1, # API 是從 0 開始索引
                        'endColumnIndex': column_index # API 是從 0 開始索引
                    },
                    'rule': {
                        'condition': {
                            'type': 'BOOLEAN',
                        },
                        'strict': True, # 強制只允許布林值
                        'showCustomUi': True # 顯示核取方塊
                    }
                }
            }
        ]

        body = {'requests': requests}
        response = service.spreadsheets().batchUpdate(spreadsheetId=sheet_id, body=body).execute()
        print(f"--- 成功為 '{worksheet_name}' 工作表的第 {column_index} 欄設定核取方塊資料驗證。---")

    except Exception as e:
        print(f"設定核取方塊資料驗證失敗：{e}")

In [132]:
# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai
client = genai.Client(api_key=api_key)

MODEL_ID = 'gemini-2.5-flash'

# (可選) 測試 AI 模型
response = client.models.generate_content(
    model = MODEL_ID, contents="Explain how AI works in a few words"
)
print(response.text)

AI learns patterns from data to make decisions.


### 定義 AI 摘要函式

In [133]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個50字的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        response = client.models.generate_content(model = MODEL_ID, contents = prompt_text)
        summary = response.text.replace('\n', ' ').strip()
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

In [134]:
def process_grades_and_summary(grade_data):
    """
    處理 Gradio 介面傳入的成績，寫入 Google Sheet 並生成 AI 摘要。
    grade_data 預期是 [科目, 成績, 是否已訂正] 的列表的列表，例如：[['國文', 90, True], ['英文', 85, False]]
    """
    global _gc, _ws, SHEET_URL, WORKSHEET_NAME, corrected_col_index

    if _ws is None:
        # 如果連線失敗，嘗試重新設定 (可能在 Gradio 介面啟動後才執行)
        setup_gspread(SHEET_URL, WORKSHEET_NAME) # Modified to pass arguments
        if _ws is None:
            return "Google Sheet 未能成功連線，請檢查錯誤訊息並重試。", ""

    if not grade_data:
        return "沒有輸入任何成績，請輸入科目、成績和訂正狀態。", ""

    # 準備寫入 Google Sheet 的成績資料，增加日期欄位和訂正狀態
    new_grades_for_sheet = []
    new_grades_for_ai_summary = [] # 準備一份不含訂正狀態的成績列表給 AI 摘要
    today = datetime.now().strftime('%Y-%m-%d')
    for record in grade_data:
        if len(record) != 3:
            return "輸入資料格式錯誤，每筆成績應包含科目、成績和是否已訂正。", ""
        subject, grade_str, is_corrected_bool = record
        try:
            grade = int(grade_str)
            # 將 Python 布林值直接傳遞，gspread 會正確處理為 Google Sheet 的布林值
            corrected_status_gspread = is_corrected_bool # 直接使用 Python 布林值
            new_grades_for_sheet.append([today, subject, grade, corrected_status_gspread])
            new_grades_for_ai_summary.append([today, subject, grade])
        except ValueError:
            return f"科目 '{subject}' 的成績 '{grade_str}' 無效，成績必須是數字。", ""

    try:
        # 將新成績寫入 Google Sheet
        _ws.append_rows(new_grades_for_sheet)
        sheet_message = "成績已成功寫入 Google Sheet。\n"

        # 根據使用者要求，在每次提交後重新設定核取方塊資料驗證
        set_checkbox_data_validation(SHEET_URL, WORKSHEET_NAME, corrected_col_index)
        sheet_message += "核取方塊資料驗證已更新。\n"

    except Exception as e:
        sheet_message = f"寫入 Google Sheet 失敗：{e}\n"
        print(f"寫入 Google Sheet 失敗：{e}")
        # 即使寫入失敗，仍嘗試生成 AI 摘要

    # 獲取 AI 摘要
    # 傳遞不含訂正狀態的成績列表給 AI，因為 AI 目前的提示不需要這項資訊
    summary = get_ai_summary(new_grades_for_ai_summary)

    try:
        # Determine the next available row more robustly
        # Get all values from the sheet to find the last row with content
        all_content_rows = _ws.get_all_values()
        next_row_for_summary = len(all_content_rows) + 1

        # The API error message indicates Max rows: 1009. We'll use this as the limit.
        # This limit might be specific to previous runs where the sheet was full.
        # For a cleared sheet, append_rows will naturally put data after existing content.
        # Let's adjust this logic to be more flexible after clearing the sheet.
        # The primary issue is the sheet being full, not a strict arbitrary limit here if cleared.
        # We will remove the MAX_SHEET_ROWS check for now, assuming the sheet is properly cleared.
        # If issues persist, we might re-evaluate this or use a larger dynamic limit.

        # Write AI summary header (date and 'AI 摘要')
        _ws.update_cell(next_row_for_summary, 1, datetime.now().strftime('%Y-%m-%d'))
        _ws.update_cell(next_row_for_summary, 2, 'AI 摘要')

        # Write summary lines (assuming AI summary is relatively short or fits in one cell)
        # If summary has newlines, we might want to put them in separate cells or handle them.
        # For now, let's just write the whole summary into the AI摘要 column in one cell.
        _ws.update_cell(next_row_for_summary, 5, summary) # Write to fifth column "AI摘要"

        sheet_message += "AI 摘要已成功寫入 Google Sheet。\n"

    except Exception as e:
        sheet_message += f"寫入 AI 摘要到 Google Sheet 失敗：{e}\n"
        print(f"寫入 AI 摘要到 Google Sheet 失敗：{e}")

    return sheet_message, summary

In [135]:
# 確保 Google Sheet 連線已經建立或重新建立
setup_gspread(SHEET_URL, WORKSHEET_NAME)

# 準備測試資料
# 每筆成績現在應該包含科目、成績和是否已訂正 (布林值)
test_grade_data = [
    ["國文", "85", False],
    ["數學", "78", True],
    ["英文", "92", False]
]

print("\n--- 正在執行 process_grades_and_summary 函式單元測試... ---")

sheet_status, ai_summary_output = process_grades_and_summary(test_grade_data)

print("\n--- 函式執行結果 --- ")
print(f"Google Sheet 處理狀態: {sheet_status}")
print(f"AI 摘要:\n{ai_summary_output}")

# 檢查 _ws 是否為 None，判斷 Google Sheet 是否真的連線成功
if _ws is None:
    print("\n注意：Google Sheet 工作表物件 (_ws) 仍為 None，表示連線可能仍有問題。")
else:
    print("\nGoogle Sheet 工作表物件 (_ws) 已成功初始化，連線似乎已建立。")


--- 正在執行 process_grades_and_summary 函式單元測試... ---
--- 正在嘗試為 '工作表2' 工作表的第 4 欄設定核取方塊資料驗證... ---


--- 成功為 '工作表2' 工作表的第 4 欄設定核取方塊資料驗證。---

--- 正在呼叫 AI 模型生成摘要... ---
寫入 AI 摘要到 Google Sheet 失敗：APIError: [400]: Range ('工作表2'!A1046) exceeds grid limits. Max rows: 1045, max columns: 26

--- 函式執行結果 --- 
Google Sheet 處理狀態: 成績已成功寫入 Google Sheet。
核取方塊資料驗證已更新。
寫入 AI 摘要到 Google Sheet 失敗：APIError: [400]: Range ('工作表2'!A1046) exceeds grid limits. Max rows: 1045, max columns: 26

AI 摘要:
好的，這是根據您提供的學生三科成績所做的總結與迷思整理：  ---  **成績摘要 (約50字):** 該學生在2026年4月23日國文、數學、英文三科考試中，成績介於78至92分。英文92分表現突出，數學78分是本次最低，國文85分。整體成績均達及格水準，顯示各科表現有差異，但整體學習穩定。  ---  **常見迷思整理:**  1.  **單次成績代表學生全貌：** 僅憑一次考試的成績，無法全面評估學生的學習潛力、努力程度或對知識的實際應用能力。學習是一個持續的過程。 2.  **成績是唯一評量標準：** 學生的學習成效不僅體現在分數上，還包括思考能力、解決問題能力、創造力、團隊合作及學習態度等，這些是成績單上不易呈現的。 3.  **科目強弱是固定不變的：** 學生在不同科目上表現的差異是常態。認為某科成績較低就代表「不擅長」該科，可能忽略了學習方法、興趣、教師教學方式等可變因素。 4.  **高分等於完全理解，低分等於一無所知：** 高分可能仍有理解不透徹之處，而低分學生可能對某些知識點仍有掌握，只是未能完全體現在考卷上。分數僅是當下表現的量化結果。 5.  **只看結果不看過程：** 過度關注成績數字，而忽略了學生在準備過程中的努力、遇到的困難、如何克服以及學習策略的調整，這對學生的長期發展和成長動力是不利的。

Google Sheet 工作表物件 (_ws) 已成功初始化，連線似乎已建立。


定義 Gradio 處理函式

In [143]:
import gradio as gr

# Define a maximum number of grade entry rows
MAX_GRADE_ROWS = 5

def _collect_and_process_grades(*args):
    """
    Helper function to collect values from individual components,
    format them, and then call the original process_grades_and_summary.
    """
    print("\n--- Gradio Callback: _collect_and_process_grades 啟動 ---")
    print(f"接收到的 Gradio 輸入 (args): {args}")

    collected_grade_data = []
    # Iterate in steps of 3 (subject, grade, corrected)
    for i in range(0, len(args), 3):
        subject = args[i]
        grade_str = args[i+1]
        is_corrected_bool = args[i+2]

        # Only process rows where at least subject or grade is provided
        if subject or grade_str:
            collected_grade_data.append([subject, grade_str, is_corrected_bool])

    print(f"收集到的成績資料: {collected_grade_data}")

    if not collected_grade_data:
        print("沒有收集到任何有效的成績輸入。")
        return "沒有輸入任何成績，請輸入科目、成績和訂正狀態。", ""

    sheet_message, summary = process_grades_and_summary(grade_data=collected_grade_data)

    print(f"process_grades_and_summary 返回的 sheet_message: {sheet_message}")
    print(f"process_grades_and_summary 返回的 summary: {summary}")
    print("--- Gradio Callback: _collect_and_process_grades 完成 ---\n")

    return sheet_message, summary


with gr.Blocks() as demo:
    gr.Markdown("# 成績輸入與 AI 摘要工具")
    gr.Markdown("請在下方的表格中輸入學生的科目、成績和是否已訂正，然後點擊『送出』。系統會將資料寫入 Google Sheet 並生成 AI 摘要。")

    # List to hold all input components for click event
    all_grade_inputs = []

    with gr.Column(): # Use a column to stack the rows of inputs
        gr.Markdown("### 成績輸入 (最多 5 筆)")
        # Create a fixed number of rows for input
        for i in range(MAX_GRADE_ROWS):
            with gr.Row():
                subject_input = gr.Textbox(label=f"科目 {i+1}", placeholder="例如：國文", scale=2)
                grade_input = gr.Textbox(label=f"成績 {i+1}", placeholder="例如：85", scale=1)
                corrected_input = gr.Checkbox(label=f"已訂正 {i+1}", value=False, scale=1)
                all_grade_inputs.extend([subject_input, grade_input, corrected_input])

        submit_button = gr.Button("送出")

    with gr.Column():
        sheet_output = gr.Textbox(label="Google Sheet 處理狀態")
        summary_output = gr.Textbox(label="AI 摘要", lines=15)

    submit_button.click(
        _collect_and_process_grades,
        inputs=all_grade_inputs,
        outputs=[sheet_output, summary_output]
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://338b956d4f98aa875c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [137]:
# 在清除並設定標題後，嘗試設定 '是否已訂正' 欄位的核取方塊資料驗證
# corrected_col_index is now global, defined in Y8awQTFN4ITU

print(f"--- 正在嘗試為 '{WORKSHEET_NAME}' 工作表的第 {corrected_col_index} 欄設定核取方塊資料驗證... ---")
set_checkbox_data_validation(SHEET_URL, WORKSHEET_NAME, corrected_col_index)
print(f"--- 成功為 '{WORKSHEET_NAME}' 工作表的第 {corrected_col_index} 欄設定核取方塊資料驗證。---")

--- 正在嘗試為 '工作表2' 工作表的第 4 欄設定核取方塊資料驗證... ---
--- 正在嘗試為 '工作表2' 工作表的第 4 欄設定核取方塊資料驗證... ---


--- 成功為 '工作表2' 工作表的第 4 欄設定核取方塊資料驗證。---
--- 成功為 '工作表2' 工作表的第 4 欄設定核取方塊資料驗證。---


In [144]:
def clear_google_sheet(sheet_url, worksheet_name):
    """
    清除指定 Google Sheet 工作表的所有內容，並將工作表行數縮減至只剩標題行。
    """
    print(f"--- 正在嘗試清除 '{worksheet_name}' 工作表的內容並縮減行數... ---")
    global _gc, _ws
    setup_gspread(sheet_url, worksheet_name) # 確保連線已建立

    if _ws is not None:
        try:
            # 1. 清除所有內容
            _ws.clear()

            # 2. 獲取當前工作表總行數
            current_row_count = _ws.row_count
            print(f"--- 當前工作表總行數: {current_row_count} ---")

            # 3. 如果行數大於 1 (標題行)，則刪除多餘的行
            if current_row_count > 1:
                # 刪除從第 2 行到最後一行的所有行
                _ws.delete_rows(2, current_row_count)
                print(f"--- 已刪除從第 2 行到第 {current_row_count} 行。---")

            # 4. 重新設定標題
            _ws.update(range_name='A1', values=[new_headers])
            print(f"--- '{worksheet_name}' 工作表已成功清除並重新設定標題。---")

            # 確保有足夠的行數讓資料驗證可以應用
            current_sheet_rows = _ws.row_count
            min_rows_for_validation = 1 # 設定一個合理的預設行數
            if current_sheet_rows < min_rows_for_validation:
                rows_to_add = min_rows_for_validation - current_sheet_rows
                _ws.add_rows(rows_to_add) # 新增行數到工作表底部
                print(f"--- 已新增 {rows_to_add} 個空行以確保資料驗證範圍。---")

            # 5. 重新設定核取方塊資料驗證
            set_checkbox_data_validation(SHEET_URL, WORKSHEET_NAME, corrected_col_index)

        except Exception as e:
            print(f"--- 清除工作表 '{worksheet_name}' 失敗：{e} ---")
    else:
        print("--- Google Sheet 連線失敗，無法執行清除操作。---")

# 確保所需變數已定義 (從 Y8awQTFN4ITU 複製，以防執行順序問題)
# Global variables for Google Sheet connection
SHEET_URL = "https://docs.google.com/spreadsheets/d/1pA05rdBicdtbP1LZQIQrls8HxSqm74JgKmyFKAqLvzw/edit?usp=drivesdk"
WORKSHEET_NAME = "工作表2"
# REQUIRED_COLUMNS is not directly used in the call, but kept for context if needed
# new_headers is used within clear_google_sheet
# corrected_col_index is used within clear_google_sheet

# 執行清除函式 (請注意：這會清空您的工作表！)
clear_google_sheet(SHEET_URL, WORKSHEET_NAME)

--- 正在嘗試清除 '工作表2' 工作表的內容並縮減行數... ---
--- 當前工作表總行數: 54 ---
--- 已刪除從第 2 行到第 54 行。---
--- '工作表2' 工作表已成功清除並重新設定標題。---
--- 正在嘗試為 '工作表2' 工作表的第 4 欄設定核取方塊資料驗證... ---


設定核取方塊資料驗證失敗：<HttpError 400 when requesting https://sheets.googleapis.com/v4/spreadsheets/1pA05rdBicdtbP1LZQIQrls8HxSqm74JgKmyFKAqLvzw:batchUpdate?alt=json returned "Invalid requests[0].setDataValidation: Range ('工作表2'!D2:D10000) exceeds grid limits. Max rows: 1, max columns: 26". Details: "Invalid requests[0].setDataValidation: Range ('工作表2'!D2:D10000) exceeds grid limits. Max rows: 1, max columns: 26">
